> Assignment using the International Debt .csv data

In [1]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName('debt-session-assignment').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/14 15:24:21 WARN Utils: Your hostname, anony-threat, resolves to a loopback address: 127.0.1.1; using 192.168.100.52 instead (on interface wlp0s20f3)
25/08/14 15:24:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/14 15:24:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/14 15:24:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
#read data from csv file

df=spark.read.csv("international_debt_with_missing_values.csv", header=True)
df.show()

+------------+------------+--------------------+--------------+-----------------+
|country_name|country_code|      indicator_name|indicator_code|             debt|
+------------+------------+--------------------+--------------+-----------------+
| Afghanistan|         AFG|Disbursements on ...|DT.DIS.DLXF.CD|       72894453.7|
| Afghanistan|        NULL|Interest payments...|DT.INT.DLXF.CD|       53239440.1|
|        NULL|         AFG|PPG, bilateral (A...|DT.AMT.BLAT.CD|       61739336.9|
| Afghanistan|         AFG|PPG, bilateral (D...|DT.DIS.BLAT.CD|       49114729.4|
| Afghanistan|         AFG|PPG, bilateral (I...|DT.INT.BLAT.CD|       39903620.1|
| Afghanistan|         AFG|PPG, multilateral...|DT.AMT.MLAT.CD|             NULL|
| Afghanistan|         AFG|                NULL|DT.DIS.MLAT.CD|       23779724.3|
| Afghanistan|         AFG|                NULL|DT.INT.MLAT.CD|       13335820.0|
| Afghanistan|         AFG|                NULL|DT.AMT.OFFT.CD|      100847181.9|
| Afghanistan|  

In [29]:
# number of rows in the dataframe...
df.count()

2357

In [7]:
df.createOrReplaceTempView("Inter_debt")

In [9]:
# 1. What is the total amount of debt owed by all countries in the dataset?
spark.sql("SELECT SUM(debt) as sum_of_debt FROM inter_debt").show()

+-------------------+
|        sum_of_debt|
+-------------------+
|2.82389330025909E12|
+-------------------+



In [12]:
# 2. How many distinct countries are recorded in the dataset?
spark.sql("SELECT DISTINCT(country_name) FROM inter_debt").show()

+--------------------+
|        country_name|
+--------------------+
|          South Asia|
|                Chad|
|            Paraguay|
|    Congo, Dem. Rep.|
|             Senegal|
|          Cabo Verde|
|Least developed c...|
|      Macedonia, FYR|
|              Guyana|
|             Eritrea|
|         Philippines|
|            Djibouti|
|               Tonga|
|                Fiji|
|              Turkey|
|              Malawi|
|             Comoros|
|         Afghanistan|
|            Cambodia|
|              Jordan|
+--------------------+
only showing top 20 rows


In [13]:
# 3. What are the distinct types of indicators and what do they represent?
spark.sql("SELECT DISTINCT(indicator_code) as code, indicator_name FROM inter_debt").show()

+--------------+--------------------+
|          code|      indicator_name|
+--------------+--------------------+
|DT.INT.MLAT.CD|PPG, multilateral...|
|DT.DIS.PRVT.CD|PPG, private cred...|
|DT.DIS.OFFT.CD|PPG, official cre...|
|DT.DIS.DLXF.CD|Disbursements on ...|
|DT.AMT.DLXF.CD|Principal repayme...|
|DT.AMT.PCBK.CD|PPG, commercial b...|
|DT.AMT.OFFT.CD|PPG, official cre...|
|DT.AMT.BLAT.CD|PPG, bilateral (A...|
|DT.DIS.BLAT.CD|PPG, bilateral (D...|
|DT.INT.BLAT.CD|PPG, bilateral (I...|
|DT.INT.PBND.CD|PPG, bonds (INT, ...|
|DT.INT.PRVT.CD|PPG, private cred...|
|DT.INT.DLXF.CD|Interest payments...|
|DT.AMT.DPNG.CD|Principal repayme...|
|DT.DIS.PROP.CD|PPG, other privat...|
|DT.AMT.PRVT.CD|PPG, private cred...|
|DT.INT.DPNG.CD|Interest payments...|
|DT.AMT.PBND.CD|PPG, bonds (AMT, ...|
|DT.INT.PROP.CD|PPG, other privat...|
|DT.DIS.MLAT.CD|PPG, multilateral...|
+--------------+--------------------+
only showing top 20 rows


In [22]:
# 4. Which country has the highest total debt and how much does it owe?

spark.sql("""SELECT country_name,country_code, SUM(debt) as total_debt 
          FROM inter_debt 
          GROUP BY country_name, country_code
          ORDER BY total_debt DESC
          LIMIT 1""").show()

+------------+------------+--------------------+
|country_name|country_code|          total_debt|
+------------+------------+--------------------+
|       China|         CHN|2.581340034560999...|
+------------+------------+--------------------+



In [27]:
# 5. What is the average debt across different debt indiactors?
spark.sql("""SELECT DISTINCT(indicator_code) as ind_code, AVG(debt) 
          FROM inter_debt 
          GROUP BY ind_code""").show()

+--------------+--------------------+
|      ind_code|           avg(debt)|
+--------------+--------------------+
|DT.AMT.DLXF.CD| 6.128831283806797E9|
|DT.DIS.PRVT.CD| 3.397664596690475E8|
|DT.INT.MLAT.CD|1.5137199686990288E8|
|DT.INT.PCBK.CD| 1.161496157391892E8|
|DT.AMT.OFFT.CD|1.2987112620897198E9|
|DT.AMT.PRVT.CD| 2.011145996591249E9|
|DT.INT.DPNG.CD|1.0323410515840578E9|
|DT.INT.PROP.CD| 2.835540396190477E7|
|DT.AMT.PROP.CD| 9.388654932625002E8|
|DT.INT.BLAT.CD|1.3240570374479164E8|
|DT.INT.PBND.CD|  8.29342457908621E8|
|DT.DIS.MLAT.CD|   6.5826154893125E8|
|DT.DIS.OFFT.CD|1.5722267505436168E9|
|DT.DIS.DLXF.CD|1.8286979421561902E9|
|DT.DIS.PCBK.CD|2.8210749199487174E8|
|DT.DIS.BLAT.CD| 1.181180190306593E9|
|DT.DIS.PROP.CD| 8.267798464999999E7|
|DT.INT.DLXF.CD|1.6063254280857148E9|
|DT.AMT.PCBK.CD| 7.101860544095241E8|
|DT.AMT.BLAT.CD|     8.19804624104E8|
+--------------+--------------------+
only showing top 20 rows


In [30]:
# 6. Which country has made the highest number of principal repayments?


In [33]:
# 7. What is the most common debt indicator across all countries?

spark.sql("""SELECT country_name, indicator_code ,COUNT(indicator_code)
          FROM inter_debt
          GROUP BY country_name, indicator_code""").show()

+------------------+--------------+---------------------+
|      country_name|indicator_code|count(indicator_code)|
+------------------+--------------+---------------------+
|           Algeria|DT.INT.OFFT.CD|                    1|
|            Brazil|DT.DIS.MLAT.CD|                    1|
|              Chad|DT.AMT.PROP.CD|                    1|
|          Djibouti|DT.DIS.OFFT.CD|                    1|
|Dominican Republic|DT.DIS.PRVT.CD|                    1|
|          Ethiopia|DT.INT.PBND.CD|                    1|
|          Ethiopia|DT.DIS.MLAT.CD|                    1|
|          Ethiopia|DT.AMT.DLXF.CD|                    1|
|          IDA only|DT.AMT.DLXF.CD|                    1|
|            Kosovo|DT.INT.DPNG.CD|                    1|
|           Nigeria|DT.DIS.OFFT.CD|                    1|
|          Paraguay|DT.AMT.DPNG.CD|                    1|
|          Tanzania|DT.DIS.MLAT.CD|                    1|
|              Togo|DT.DIS.DLXF.CD|                    1|
|           Vi

In [34]:
# 8. Identify any other key debt trends and summarize your findings
